Metode Backward Chaining adalah *Goal-driven reasoning* metode logika dari goals (kesimpulan) baru dibuktikan dengan Data.
Tujuannya pada sistem ini ada 2 yaitu :
1.   Menetentukan risk level
2.   Menentukan rekomendasi belajar

Dengan cara menentukan hipotesis terlebih dahulu, kemudian sistem akan memeriksa apakah kondisi yang dimiliki siswa memenuhi rule yang telah ditentukan untuk mendukung kebenaran dari hipotesis tersebut.

#Prepocessing Dataset

In [ ]:
#Upload dataset
from google.colab import files
uploaded = files.upload()

Saving StudentPerformanceFactors.csv to StudentPerformanceFactors.csv


In [ ]:
#Load dataset
import pandas as pd

df = pd.read_csv('StudentPerformanceFactors.csv')
df.head()

,Hours_Studied,Attendance,Parental_Involvement,Access_to_Resources,Extracurricular_Activities,Sleep_Hours,Previous_Scores,Motivation_Level,Internet_Access,Tutoring_Sessions,Family_Income,Teacher_Quality,School_Type,Peer_Influence,Physical_Activity,Learning_Disabilities,Parental_Education_Level,Distance_from_Home,Gender,Exam_Score
0,23,84,Low,High,No,7,73,Low,Yes,0,Low,Medium,Public,Positive,3,No,High School,Near,Male,67
1,19,64,Low,Medium,No,8,59,Low,Yes,2,Medium,Medium,Public,Negative,4,No,College,Moderate,Female,61
2,24,98,Medium,Medium,Yes,7,91,Medium,Yes,2,Medium,Medium,Public,Neutral,4,No,Postgraduate,Near,Male,74
3,29,89,Low,Medium,Yes,8,98,Medium,Yes,1,Medium,Medium,Public,Negative,4,No,High School,Moderate,Male,71
4,19,92,Medium,Medium,Yes,6,65,Medium,Yes,3,Medium,High,Public,Neutral,4,No,College,Near,Female,70


A. Data selection

In [ ]:
#ambil 8 variabel sesuai kebutuhan sistem
data = df[[
    'Hours_Studied',
    'Parental_Involvement',
    'Access_to_Resources',
    'Sleep_Hours',
    'Previous_Scores',
    'Motivation_Level',
    'Tutoring_Sessions',
    'Learning_Disabilities'
]].copy()

data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6607 entries, 0 to 6606
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   Hours_Studied          6607 non-null   int64 
 1   Parental_Involvement   6607 non-null   object
 2   Access_to_Resources    6607 non-null   object
 3   Sleep_Hours            6607 non-null   int64 
 4   Previous_Scores        6607 non-null   int64 
 5   Motivation_Level       6607 non-null   object
 6   Tutoring_Sessions      6607 non-null   int64 
 7   Learning_Disabilities  6607 non-null   object
dtypes: int64(4), object(4)
memory usage: 413.1+ KB


B. Data Cleaning

In [ ]:
#cek null value
data.isnull().sum()

,0
Hours_Studied,0
Parental_Involvement,0
Access_to_Resources,0
Sleep_Hours,0
Previous_Scores,0
Motivation_Level,0
Tutoring_Sessions,0
Learning_Disabilities,0


tidak ditemukan null value pada dataset, lanjut pada pengecekan berikutnya

In [ ]:
#cek duplikat data
data.duplicated().sum()

np.int64(49)

ditemukan 49 data duplikat pada dataset, sehingga perlu dilakukan proses penghapusan data duplikat untuk menghindari bias dalam analisis




In [ ]:
#hapus duplikat data
data = data.drop_duplicates()

In [ ]:
#cek kembali
data.duplicated().sum()

np.int64(0)

In [ ]:
#cek outlier (nilai aneh)
data.describe()

,Hours_Studied,Sleep_Hours,Previous_Scores,Tutoring_Sessions
count,6558.000000,6558.000000,6558.000000,6558.000000
mean,19.979872,7.029887,75.039646,1.495883
std,6.002228,1.470888,14.396135,1.232434
min,1.000000,4.000000,50.000000,0.000000
25%,16.000000,6.000000,63.000000,1.000000
50%,20.000000,7.000000,75.000000,1.000000
75%,24.000000,8.000000,88.000000,2.000000
max,44.000000,10.000000,100.000000,8.000000


tidak ditemukan data outlier pada tiap variabel yang digunakan. Data bersih dan rapi siap untuk diproses

C. Data Transformation

In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6558 entries, 0 to 6606
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   Hours_Studied          6558 non-null   int64 
 1   Parental_Involvement   6558 non-null   object
 2   Access_to_Resources    6558 non-null   object
 3   Sleep_Hours            6558 non-null   int64 
 4   Previous_Scores        6558 non-null   int64 
 5   Motivation_Level       6558 non-null   object
 6   Tutoring_Sessions      6558 non-null   int64 
 7   Learning_Disabilities  6558 non-null   object
dtypes: int64(4), object(4)
memory usage: 461.1+ KB


In [ ]:
#Ubah semua data jadi Kategorikal (Low/Medium/High)
#Hours Studied
def cat_hours(x):
    if x > 7:
        return 'LOW'
    elif 5 <= x <= 7:
        return 'MEDIUM'
    else:
        return 'HIGH'

#Previous Scores
def cat_scores(x):
    if x >= 75:
        return 'LOW'
    elif 60 <= x <= 74:
        return 'MEDIUM'
    else:
        return 'HIGH'

#Sleep Hours
def cat_sleep(x):
    if x > 8:
        return 'LOW'
    elif 6 <= x <= 8:
        return 'MEDIUM'
    else:
        return 'HIGH'

#Text (High/Medium/Low)
#Fungsi ini dipakai untuk membalik keadaan (ubah kategori kondisi → jadi kategori risiko) pada variabel Parental_Involvement, Access_to_Resources, dan Motivation_Level
def cat_text(x):
    if x == 'High':
        return 'LOW'
    elif x == 'Medium':
        return 'MEDIUM'
    else:
        return 'HIGH'

#Tutoring
def cat_tutor(x):
    if x >= 2:
        return 'LOW'
    elif x == 1:
        return 'MEDIUM'
    else:
        return 'HIGH'

#Learning Disabilities
#Pada dataset cuma Yes/No, sehingga kita asumsikan No → LOW (tidak ada) dan Yes → MEDIUM (ringan)
def cat_disability(x):
    if x == 'No':
        return 'LOW'
    else:
        return 'MEDIUM'

#Apply Rule-Based ke data
data['hours_cat'] = data['Hours_Studied'].apply(cat_hours)
data['score_cat'] = data['Previous_Scores'].apply(cat_scores)
data['sleep_cat'] = data['Sleep_Hours'].apply(cat_sleep)
data['parent_cat'] = data['Parental_Involvement'].apply(cat_text)
data['resource_cat'] = data['Access_to_Resources'].apply(cat_text)
data['motivation_cat'] = data['Motivation_Level'].apply(cat_text)
data['tutor_cat'] = data['Tutoring_Sessions'].apply(cat_tutor)
data['disability_cat'] = data['Learning_Disabilities'].apply(cat_disability)

In [ ]:
#Cek data perubahan, harus muncul kolom baru
data.head()

,Hours_Studied,Parental_Involvement,Access_to_Resources,Sleep_Hours,Previous_Scores,Motivation_Level,Tutoring_Sessions,Learning_Disabilities,hours_cat,score_cat,sleep_cat,parent_cat,resource_cat,motivation_cat,tutor_cat,disability_cat
0,23,Low,High,7,73,Low,0,No,LOW,MEDIUM,MEDIUM,HIGH,LOW,HIGH,HIGH,LOW
1,19,Low,Medium,8,59,Low,2,No,LOW,HIGH,MEDIUM,HIGH,MEDIUM,HIGH,LOW,LOW
2,24,Medium,Medium,7,91,Medium,2,No,LOW,LOW,MEDIUM,MEDIUM,MEDIUM,MEDIUM,LOW,LOW
3,29,Low,Medium,8,98,Medium,1,No,LOW,LOW,MEDIUM,HIGH,MEDIUM,MEDIUM,MEDIUM,LOW
4,19,Medium,Medium,6,65,Medium,3,No,LOW,MEDIUM,MEDIUM,MEDIUM,MEDIUM,MEDIUM,LOW,LOW


Sudah ada kolom baru untuk rule-based yang kita buat berarti sudah apply

#Inferensi Rule-Based

In [ ]:
#1. Menentukan Tingkat Resiko Akademik
def hitung_risiko(row):
    values = [
        row['hours_cat'],
        row['score_cat'],
        row['sleep_cat'],
        row['motivation_cat'],
        row['parent_cat'],
        row['resource_cat'],
        row['tutor_cat'],
        row['disability_cat']
    ]

    high = values.count('HIGH')
    medium = values.count('MEDIUM')
    low = values.count('LOW')

    #Rule-Based Resiko
    if high >= 5:
        return 'HIGH'
    elif high >= 3 and medium >= 2:
        return 'HIGH'
    elif high == 1 and medium >= 2:
        return 'MEDIUM'
    elif high >= 2 and low == 0:
        return 'MEDIUM'
    elif medium >= 4:
        return 'MEDIUM'
    elif low == 8:
        return 'LOW'
    elif low > high:
        return 'LOW'
    else:
        return 'MEDIUM'

In [ ]:
    #Apply Rule-Based ke Data
    data['risk_level'] = data.apply(hitung_risiko, axis=1)

In [ ]:
#Cek Hasil
data[['hours_cat','score_cat','risk_level']].head()

,hours_cat,score_cat,risk_level
0,LOW,MEDIUM,HIGH
1,LOW,HIGH,HIGH
2,LOW,LOW,MEDIUM
3,LOW,LOW,MEDIUM
4,LOW,MEDIUM,MEDIUM


ex : data kolom 0, artinya =
hours_cat = LOW → jam belajar bagus (≥ 7 jam/minggu)
score_cat = MEDIUM → nilai sedang
tapi, risk_level = HIGH → risiko akademik tinggi
karena risk_level TIDAK cuma dari 2 variabel (hours_cat & score_cat) ini
Sehingga walaupun jam belajar bagus, nilai juga cukup bagus, kemungkinan motivasi rendah / tidak pernah les / orangtua kurang terlibat dalam kegiatan belajar hasilnya HIGH RISK

In [ ]:
#2. Menentukan Rekomendasi Teknik Belajar
def rekomendasi_belajar(row):

    h = row['hours_cat']
    s = row['sleep_cat']
    sc = row['score_cat']
    m = row['motivation_cat']
    p = row['parent_cat']
    r = row['resource_cat']
    t = row['tutor_cat']
    d = row['disability_cat']
    risk = row['risk_level']

    # RULE 1
    if s == 'MEDIUM' and h == 'LOW' and sc == 'LOW' and d == 'MEDIUM':
        return "Tidak suka belajar lama tapi cukup paham | Pomodoro, Active Recall, Blurting"

    # RULE 2
    elif s == 'HIGH' and h == 'HIGH' and m == 'HIGH':
        return "Mudah lelah & kurang fokus | Pomodoro, Interleaving, Mind Mapping"

    # RULE 3
    elif h == 'LOW' and m == 'LOW' and sc == 'MEDIUM':
        return "Rajin tapi kurang efektif | Feynman, Active Recall, Blurting"

    # RULE 4
    elif m == 'HIGH' and p == 'HIGH' and t == 'HIGH':
        return "Kurang dorongan belajar | Interleaving, Pomodoro, Mind Mapping"

    # RULE 5
    elif r == 'HIGH' and t == 'HIGH':
        return "Keterbatasan fasilitas | SQ3R, Feynman, Teach Back"

    # RULE 6
    elif s == 'LOW' and m == 'LOW' and h == 'LOW':
        return "Kondisi optimal | Active Recall, Spaced Repetition, Feynman"

    # RULE 7
    elif d == 'HIGH' and h == 'HIGH':
        return "Sulit memahami materi | Mind Mapping, Teach Back, Feynman"

    # RULE 8
    elif sc == 'HIGH' and m == 'MEDIUM' and h == 'MEDIUM':
        return "Pemahaman kurang stabil | Spaced Repetition, Active Recall, Blurting"

    # RULE 9
    elif t == 'LOW' and m == 'MEDIUM':
        return "Sudah terbantu tapi kurang mandiri | Active Recall, Blurting, Feynman"

    # RULE 10
    elif p == 'LOW' and m == 'HIGH':
        return "Tergantung orang lain | Pomodoro, Interleaving, Teach Back"

    # RULE 11
    elif s == 'HIGH' and d == 'MEDIUM':
        return "Konsentrasi terganggu | Pomodoro, Mind Mapping, Spaced Repetition"

    # RULE 12
    elif h == 'HIGH' and m == 'MEDIUM' and r == 'LOW':
        return "Potensi ada tapi kurang usaha | Pomodoro, Active Recall, Interleaving"

    # DEFAULT
    else:
        if risk == 'HIGH':
            return "Butuh perbaikan serius | Spaced Repetition, Pomodoro, Active Recall"
        elif risk == 'MEDIUM':
            return "Perlu peningkatan belajar | Active Recall, Feynman, Interleaving"
        else:
            return "Pertahankan performa | Active Recall, Spaced Repetition"

In [ ]:
#Apply Rule-Based ke Data
data['rekomendasi'] = data.apply(rekomendasi_belajar, axis=1)

In [ ]:
#Cek Rekomendasi
data[['risk_level','rekomendasi']].head()

,risk_level,rekomendasi
0,HIGH,"Kurang dorongan belajar | Interleaving, Pomodo..."
1,HIGH,"Butuh perbaikan serius | Spaced Repetition, Po..."
2,MEDIUM,Sudah terbantu tapi kurang mandiri | Active Re...
3,MEDIUM,"Perlu peningkatan belajar | Active Recall, Fey..."
4,MEDIUM,Sudah terbantu tapi kurang mandiri | Active Re...


#Keterangan :
* Baris 0 = HIGH → siswa memiliki motivasi rendah dan kurangnya dukungan orangtua, maka sistem menyimpulkan : *“kurang dorongan belajar”* sehingga solusi yang bisa digunakan : Interleaving dan Pomodoro
*  Baris 1 = HIGH → siswa memiliki kondisi yang cukup buruk karena banyak faktor HIGH, jadi sistem kasih rekomendasi umum (fallback)
*   Baris 2 = MEDIUM → siswa sudah ikut les, tapi belum bisa belajar secara mandiri. Solusi : Teknik Active Recall dan Blurting

#Kelebihan & Kekurangan
Metode backward chaining memiliki kelebihan efisiensi dan fokus pada tujuan, sehingga cocok digunakan dalam sistem berbasis aturan seperti sistem rekomendasi belajar (EduTrace). Selain itu, metode ini bersifat transparan karena setiap keputusan dapat ditelusuri berdasarkan rule yang digunakan. Namun, metode ini juga memiliki keterbatasan, yaitu sangat bergantung pada kelengkapan rule dan tidak mampu belajar secara otomatis dari data, sehingga kurang fleksibel terhadap perubahan kondisi dan data.

#Kesimpulan
Metode backward chaining pada sistem ini digunakan dengan pendekatan berbasis tujuan (goal-driven), yaitu dimulai dari penentuan hipotesis berupa tingkat risiko akademik dan rekomendasi teknik belajar, kemudian sistem menelusuri apakah kondisi data siswa memenuhi aturan yang telah ditentukan.

Berdasarkan implementasi yang dilakukan, backward chaining mampu menentukan tingkat risiko akademik secara sistematis melalui pengecekan kombinasi kategori variabel, serta menghasilkan rekomendasi belajar yang sesuai dengan kondisi masing-masing siswa. Proses inferensi berjalan secara terarah karena hanya berfokus pada tujuan yang ingin dicapai, sehingga lebih efisien dan mudah ditelusuri.

Dengan demikian, metode backward chaining terbukti efektif untuk digunakan dalam sistem rekomendasi berbasis aturan, khususnya dalam kasus analisis performa siswa, karena mampu memberikan keputusan yang logis, transparan, dan sesuai dengan rule yang telah dirancang.